In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/brand-buzzword-hackathon/sample_submission.csv
/kaggle/input/competitions/brand-buzzword-hackathon/test.txt
/kaggle/input/competitions/brand-buzzword-hackathon/train.txt


In [2]:
"""Validated Transformer + train-only pattern-candidate posterior.

Start with NUM_MODELS=1.  This script evaluates whether a candidate posterior
actually improves held-out words before spending time on a two-model ensemble.
"""
from __future__ import annotations

import csv
import random
from collections import defaultdict
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

ALPHABET = "abcdefghijklmnopqrstuvwxyz"
PAD, MASK, MAX_LEN = 0, 1, 29
NUM_MODELS = 2
EPOCHS = 10
VALIDATION_WORDS = 12_000
CANDIDATE_WEIGHTS = (0.0, 0.10, 0.20, 0.35, 0.50, 0.65)
# Extra decision-layer weight.  It rewards guesses whose hit/miss reveal pattern
# would split the candidate posterior; zero is the existing 52.15% policy.
DECISION_WEIGHTS = (0.0, 0.02, 0.05, 0.10, 0.15, 0.25)


def read_words(path):
    with open(path, encoding="utf-8") as f:
        return [x.strip().lower() for x in f if x.strip()]


def encode(word):
    x = np.full(MAX_LEN, PAD, dtype=np.int64)
    x[:len(word)] = [ord(c) - 95 for c in word]
    return x


class States(Dataset):
    def __init__(self, words, count=450_000, seed=2026):
        self.full = [encode(w) for w in words]
        self.letters = [list({ord(c) - 97 for c in w}) for w in words]
        self.count, self.seed = count, seed

    def __len__(self): return self.count

    def __getitem__(self, index):
        r = random.Random(self.seed + index)
        i = r.randrange(len(self.full)); full, letters = self.full[i], self.letters[i]
        reveal_n = min(len(letters) - 1, int((r.random() ** 1.7) * len(letters)))
        revealed = set(r.sample(letters, reveal_n)) if reveal_n else set()
        board = full.copy()
        for p, token in enumerate(board):
            if token >= 2 and token - 2 not in revealed: board[p] = MASK
        return torch.from_numpy(board), torch.from_numpy(full)


class PositionTransformer(nn.Module):
    # Same core as the 51.73% validated ensemble.
    def __init__(self, width=192, layers=5):
        super().__init__()
        self.token = nn.Embedding(28, width, padding_idx=PAD)
        self.position = nn.Embedding(MAX_LEN, width)
        block = nn.TransformerEncoderLayer(width, nhead=6, dim_feedforward=width * 3,
                                           dropout=0.10, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(block, num_layers=layers)
        self.norm = nn.LayerNorm(width)
        self.output = nn.Linear(width, 26)

    def forward(self, board):
        p = torch.arange(board.shape[1], device=board.device)
        x = self.token(board) + self.position(p)[None]
        x = self.encoder(x, src_key_padding_mask=(board == PAD))
        return self.output(self.norm(x))


class CandidateIndex:
    """Exact pattern posterior backed only by fit-word arrays and inverted lists."""
    def __init__(self, words):
        grouped = defaultdict(list)
        for w in words: grouped[len(w)].append(w)
        self.codes, self.masks, self.inverted = {}, {}, {}
        self.cache = {}
        self.calls = self.nonempty = 0
        for length, group in grouped.items():
            codes = np.asarray([[ord(c) - 95 for c in w] for w in group], dtype=np.uint8)
            masks = np.zeros(len(group), dtype=np.uint32)
            for i, row in enumerate(codes):
                masks[i] = sum(1 << (int(c) - 2) for c in set(row))
            self.codes[length], self.masks[length] = codes, masks
            for pos in range(length):
                for token in range(2, 28):
                    ids = np.flatnonzero(codes[:, pos] == token)
                    if len(ids): self.inverted[(length, pos, token)] = ids

    def reset_stats(self):
        self.calls = self.nonempty = 0
        self.cache.clear()

    def posterior(self, board: np.ndarray, missed_mask: int):
        """Return (P(letter present), candidate count), or None if uninformative."""
        known = np.flatnonzero(board >= 2)
        # One revealed character is intentionally neural-only: matching one
        # position has weak lexical specificity and creates too much work.
        if len(known) < 2: return None
        length = int(np.count_nonzero(board != PAD))
        key = (bytes(board[:length]), missed_mask)
        if key in self.cache: return self.cache[key]
        self.calls += 1
        choices = []
        for pos in known:
            ids = self.inverted.get((length, int(pos), int(board[pos])))
            if ids is None:
                self.cache[key] = None
                return None
            choices.append(ids)
        # Begin with the rarest revealed-position match, then vector-filter it.
        ids = min(choices, key=len).copy()
        codes, masks = self.codes[length], self.masks[length]
        for pos in known:
            ids = ids[codes[ids, pos] == board[pos]]
            if not len(ids):
                self.cache[key] = None
                return None
        if missed_mask:
            ids = ids[(masks[ids] & np.uint32(missed_mask)) == 0]
        n = len(ids)
        if not n:
            self.cache[key] = None
            return None
        self.nonempty += 1
        selected = masks[ids]
        p = np.asarray([np.mean((selected & np.uint32(1 << j)) != 0) for j in range(26)], dtype=np.float32)
        # For each possible guess, the feedback is its full position pattern
        # (including an all-zero pattern for a miss).  Its normalized entropy is
        # the expected candidate-set reduction supplied by that legal feedback.
        information = np.zeros(26, dtype=np.float32)
        if n > 1:
            bit_values = (np.uint32(1) << np.arange(length, dtype=np.uint32))
            selected_codes = codes[ids]
            normalizer = np.log(n)
            for j in range(26):
                patterns = ((selected_codes == j + 2).astype(np.uint32) * bit_values).sum(axis=1)
                _unique, counts = np.unique(patterns, return_counts=True)
                q = counts / n
                information[j] = float(-(q * np.log(q)).sum() / normalizer)
        # Very tiny candidate groups are fragile for unseen words; very large
        # groups add little beyond the neural prior. Confidence gates both ends.
        confidence = min(1.0, n / 20.0) * max(0.0, 1.0 - max(0, n - 5000) / 5000.0)
        result = (p, information, n, confidence)
        if len(self.cache) < 200_000: self.cache[key] = result
        return result


@torch.no_grad()
def simulate(words, models, candidate_index, device, candidate_weight=0.0, decision_weight=0.0, batch_size=4096):
    n = len(words); truth = np.stack([encode(w) for w in words])
    present = np.zeros((n, 26), dtype=bool)
    for i, w in enumerate(words): present[i, [ord(c) - 97 for c in set(w)]] = True
    board = np.where(truth == PAD, PAD, MASK).astype(np.int64)
    guessed = np.zeros((n, 26), dtype=bool)
    missed = np.zeros((n, 26), dtype=bool)
    wrong = np.zeros(n, dtype=np.int8); sequences = [[] for _ in words]
    for m in models: m.eval()
    for _ in range(26):
        alive = (board == MASK).any(1) & (wrong < 6)
        if not alive.any(): break
        ids = np.flatnonzero(alive); picks = np.empty(len(ids), dtype=np.int64)
        for start in range(0, len(ids), batch_size):
            ix = ids[start:start + batch_size]
            b = torch.as_tensor(board[ix], device=device)
            blank = (b == MASK).unsqueeze(-1); score = None
            for model in models:
                probs = model(b).softmax(-1)
                p_any = 1 - torch.exp((torch.log1p(-probs.clamp_max(1 - 1e-6)) * blank).sum(1))
                score = p_any if score is None else score + p_any
            score = (score / len(models)).cpu().numpy()
            if candidate_weight:
                for local, row in enumerate(ix):
                    # This is game feedback already observed on earlier turns;
                    # it is not derived from any unobserved test-word letters.
                    missed_mask = sum(1 << j for j in np.flatnonzero(missed[row]))
                    result = candidate_index.posterior(board[row], missed_mask)
                    if result is not None:
                        posterior, information, _count, confidence = result
                        w = candidate_weight * confidence
                        score[local] = (1 - w) * score[local] + w * posterior
                        score[local] += decision_weight * confidence * information
            score[guessed[ix]] = -np.inf
            picks[start:start + len(ix)] = score.argmax(1)
        for row, letter in zip(ids, picks):
            guessed[row, letter] = True; sequences[row].append(ALPHABET[letter])
            if present[row, letter]: board[row, truth[row] == letter + 2] = letter + 2
            else:
                wrong[row] += 1
                missed[row, letter] = True
    wins = ~(board == MASK).any(1)
    return ["".join(s) for s in sequences], float(wins.mean()), float(wrong.mean())


def train_one(fit, val, index, device, seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = PositionTransformer().to(device)
    data = States(fit, seed=seed)
    loader = DataLoader(data, batch_size=640, shuffle=True, num_workers=2,
                        persistent_workers=True, pin_memory=(device.type == "cuda"))
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.015)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=2e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    best_state, best_win = None, -1.0
    for epoch in range(EPOCHS):
        model.train(); total = 0.0
        for board, target in loader:
            board, target = board.to(device), target.to(device); masked = board == MASK
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                loss = nn.functional.cross_entropy(model(board)[masked], target[masked] - 2)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); total += loss.item()
        schedule.step()
        _, win, wrong = simulate(val, [model], index, device)
        print(f"seed {seed} epoch {epoch+1}/{EPOCHS}: loss={total/len(loader):.4f}, heldout_win={win:.4%}, wrong={wrong:.3f}")
        if win > best_win:
            best_win = win; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_win


def make_submission(train_path, test_path, output_path="submission.csv"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train, test = read_words(train_path), read_words(test_path)
    assert len(test) == 250_000 and max(map(len, train + test)) <= MAX_LEN
    r = random.Random(2026); ids = list(range(len(train))); r.shuffle(ids)
    val = [train[i] for i in ids[:VALIDATION_WORDS]]; fit = [train[i] for i in ids[VALIDATION_WORDS:]]
    print(f"device: {device}; building fit-only candidate index...")
    index = CandidateIndex(fit)
    print(f"fit={len(fit):,}; validation={len(val):,}; test={len(test):,}")
    models = []; singles = []
    for i in range(NUM_MODELS):
        model, score = train_one(fit, val, index, device, 2026 + 7919*i)
        models.append(model); singles.append(score)
        print(f"seed {2026 + 7919*i} best neural-only={score:.4%}")
    # Candidate blending was already strongest near .35. Hold it fixed here so
    # this run isolates the decision rule instead of retuning two variables.
    selected_candidate_weight = 0.35
    trials = []
    for decision_weight in DECISION_WEIGHTS:
        index.reset_stats()
        _, win, wrong = simulate(val, models, index, device, selected_candidate_weight, decision_weight)
        coverage = index.nonempty / max(1, index.calls)
        print(f"decision_weight={decision_weight:.2f}: heldout_win={win:.4%}, wrong={wrong:.3f}, candidate_coverage={coverage:.2%}")
        trials.append((win, -wrong, decision_weight))
    best_decision_weight = max(trials)[2]
    print(f"selected candidate_weight={selected_candidate_weight:.2f}; selected decision_weight={best_decision_weight:.2f}")
    guesses, _, _ = simulate(test, models, index, device, selected_candidate_weight, best_decision_weight)
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f); writer.writerow(["word_id", "guessed_letters_string"])
        writer.writerows(enumerate(guesses))
    print(f"wrote {len(guesses):,} rows to {output_path}")


if __name__ == "__main__":
    DATA_DIR = "/kaggle/input/competitions/brand-buzzword-hackathon"
    make_submission(f"{DATA_DIR}/train.txt", f"{DATA_DIR}/test.txt")


device: cuda; building fit-only candidate index...
fit=213,300; validation=12,000; test=250,000


/tmp/ipykernel_23/676315061.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(block, num_layers=layers)


seed 2026 epoch 1/10: loss=2.5887, heldout_win=38.2083%, wrong=4.983
seed 2026 epoch 2/10: loss=2.4884, heldout_win=42.4917%, wrong=4.843
seed 2026 epoch 3/10: loss=2.4588, heldout_win=44.9250%, wrong=4.735
seed 2026 epoch 4/10: loss=2.4389, heldout_win=46.2250%, wrong=4.678
seed 2026 epoch 5/10: loss=2.4223, heldout_win=47.4083%, wrong=4.624
seed 2026 epoch 6/10: loss=2.4082, heldout_win=49.0083%, wrong=4.574
seed 2026 epoch 7/10: loss=2.3939, heldout_win=49.7917%, wrong=4.523
seed 2026 epoch 8/10: loss=2.3819, heldout_win=50.1417%, wrong=4.512
seed 2026 epoch 9/10: loss=2.3707, heldout_win=50.7333%, wrong=4.488
seed 2026 epoch 10/10: loss=2.3643, heldout_win=51.1250%, wrong=4.468
seed 2026 best neural-only=51.1250%


/tmp/ipykernel_23/676315061.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(block, num_layers=layers)


seed 9945 epoch 1/10: loss=2.5811, heldout_win=39.5833%, wrong=4.953
seed 9945 epoch 2/10: loss=2.4861, heldout_win=42.3750%, wrong=4.817
seed 9945 epoch 3/10: loss=2.4585, heldout_win=45.5583%, wrong=4.726
seed 9945 epoch 4/10: loss=2.4397, heldout_win=46.3167%, wrong=4.681
seed 9945 epoch 5/10: loss=2.4241, heldout_win=47.6833%, wrong=4.627
seed 9945 epoch 6/10: loss=2.4100, heldout_win=48.8917%, wrong=4.569
seed 9945 epoch 7/10: loss=2.3964, heldout_win=49.7667%, wrong=4.526
seed 9945 epoch 8/10: loss=2.3848, heldout_win=50.8833%, wrong=4.486
seed 9945 epoch 9/10: loss=2.3749, heldout_win=50.4083%, wrong=4.490
seed 9945 epoch 10/10: loss=2.3686, heldout_win=50.6500%, wrong=4.479
seed 9945 best neural-only=50.8833%
decision_weight=0.00: heldout_win=52.1500%, wrong=4.421, candidate_coverage=42.18%
decision_weight=0.02: heldout_win=52.1750%, wrong=4.421, candidate_coverage=42.20%
decision_weight=0.05: heldout_win=52.1083%, wrong=4.422, candidate_coverage=42.22%
decision_weight=0.10: he